In [ ]:
# !pip install gensim torch transformers scikit-learn nltk matplotlib seaborn pandas
!pip install gensim

import gensim
from gensim.models import Word2Vec, KeyedVectors
import gensim.downloader as api

import nltk
import string
import numpy as np
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

# Download corpus
nltk.download('brown', quiet=True)
from nltk.corpus import brown

print(f"Gensim version: {gensim.__version__}")
print("Setup complete.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 33.8 MB/s eta 0:00:00
Gensim version: 4.4.0
Setup complete.


In [ ]:
# Define punctuation set
punct = set(string.punctuation)

# Preprocess:
# - lowercase
# - remove punctuation
# - keep tokenized sentences
sentences = [
    [w.lower() for w in sent if w.lower() not in punct]
    for sent in brown.sents()
]

# Basic stats
print(f"Total sentences: {len(sentences):,}")
print(f"Sample sentence: {sentences[0]}")

# Flatten tokens
all_words = [w for sent in sentences for w in sent]
print(f"Total tokens: {len(all_words):,}")
print(f"Unique words: {len(set(all_words)):,}")

Total sentences: 57,340
Sample sentence: ['the', 'fulton', 'county', 'grand', 'jury', 'said', 'friday', 'an', 'investigation', 'of', "atlanta's", 'recent', 'primary', 'election', 'produced', '``', 'no', 'evidence', "''", 'that', 'any', 'irregularities', 'took', 'place']
Total tokens: 1,034,378
Unique words: 49,800


In [ ]:
# Train Word2Vec model
w2v_model = Word2Vec(
    sentences=sentences, # sentences in token format
    vector_size=100,   # embedding dimension
    window=5,          # context window
    min_count=5,       # ignore rare words
    sg=0,              # CBOW model (0 = CBOW, 1 = Skip-gram)
    workers=2,
    seed=42
)

print(f"Vocabulary size: {len(w2v_model.wv):,}")
print(f"Embedding dimension: {w2v_model.vector_size}")

Vocabulary size: 14,209
Embedding dimension: 100


In [ ]:
# Save trained model
w2v_model.save("word2vec_brown.model")
print("Model saved successfully.")

Model saved successfully.


In [ ]:
# Query words
query_words = ['man', 'house', 'love']

for word in query_words:
    if word in w2v_model.wv:
        print(f"\nMost similar to '{word}':")
        for sim_word, score in w2v_model.wv.most_similar(word, topn=5):
            print(f"  {sim_word:<15} {score:.4f}")
    else:
        print(f"{word} not in vocabulary")



Most similar to 'man':
  girl            0.8647
  woman           0.8572
  boy             0.8200
  child           0.7731
  young           0.7640

Most similar to 'house':
  hall            0.8919
  room            0.8854
  door            0.8851
  office          0.8835
  car             0.8830

Most similar to 'love':
  grateful        0.8980
  god             0.8915
  trouble         0.8909
  read            0.8836
  thinking        0.8819


**Do the similarities make sense?**

**man:**

Related to gender/people: woman, girl, boy: correct

child, young: contextual

**house:**

door, hall, room: good

office: reasonable

car: less accurate (likely context overlap in corpus)

**love:**

grateful, God, trouble, read, thinking

not semantically similar but likely co-occur frequently

Reflects context similarity instead of meaning similarity


In [ ]:
# Analogy tests
analogy_tests = [
    (['king', 'woman'], ['man'], 'king - man + woman'),
    (['paris', 'italy'], ['france'], 'paris - france + italy'),
    (['good', 'better'], ['bad'], 'good - bad + better')
]

for pos, neg, desc in analogy_tests:
    if all(w in w2v_model.wv for w in pos + neg):
        print(f"\n{desc}:")
        results = w2v_model.wv.most_similar(
            positive=pos,
            negative=neg,
            topn=3
        )
        for word, score in results:
            print(f"  -> {word} ({score:.4f})")
    else:
        print(f"{desc}: Skipped (missing vocabulary)")


king - man + woman:
  -> pupils (0.9382)
  -> memory (0.9376)
  -> youth (0.9355)

paris - france + italy:
  -> rome (0.9449)
  -> october (0.9422)
  -> august (0.9390)

good - bad + better:
  -> more (0.7752)
  -> rather (0.7277)
  -> less (0.6982)


**Do the analogies make sense?**

**king - man + woman**

incorrect

**paris - france + italy**

rome: correct

**good - bad + better**

more, rather, less: weak semantic reasoning

In [ ]:
print("Loading GloVe embeddings...")
glove_vectors = api.load("glove-wiki-gigaword-100")

print(f"GloVe vocab size: {len(glove_vectors):,}")
print(f"Embedding dimension: {glove_vectors.vector_size}")

Loading GloVe embeddings...
[==================================================] 100.0% 128.1/128.1MB downloaded
GloVe vocab size: 400,000
Embedding dimension: 100


In [ ]:
# Similarity
print("\n=== GloVe Similarity ===")
for word in query_words:
    if word in glove_vectors:
        print(f"\nMost similar to '{word}':")
        for w, s in glove_vectors.most_similar(word, topn=5):
            print(f"  {w:<15} {s:.4f}")

# Analogy
print("\n=== GloVe Analogy ===")
if all(w in glove_vectors for w in ['king', 'man', 'woman']):
    results = glove_vectors.most_similar(
        positive=['king', 'woman'],
        negative=['man'],
        topn=3
    )
    for word, score in results:
        print(f"  -> {word} ({score:.4f})")


=== GloVe Similarity ===

Most similar to 'man':
  woman           0.8323
  boy             0.7915
  one             0.7789
  person          0.7527
  another         0.7522

Most similar to 'house':
  office          0.7582
  senate          0.7205
  room            0.7150
  houses          0.6888
  capitol         0.6852

Most similar to 'love':
  me              0.7383
  passion         0.7352
  my              0.7327
  life            0.7288
  dream           0.7268

=== GloVe Analogy ===
  -> queen (0.7699)
  -> monarch (0.6843)
  -> throne (0.6756)


**Comparison:**

GloVe is better in every aspect.

**Reasons**

1. GloVe has bigger data size
2. GloVe captures relationships more globally than locally
3. GloVe learns general language structure, word2vec dataset-specific patterns
4. Noise in narratives and mixed topics of Brown corpus


In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

import numpy as np
import pandas as pd
import nltk

# from nltk.corpus import movie_reviews
from sklearn.metrics import accuracy_score, f1_score, classification_report
from sklearn.model_selection import train_test_split

import kagglehub
import os

from collections import Counter

# Download dataset
# nltk.download('movie_reviews', quiet=True)

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# Download dataset from KaggleHub
path = kagglehub.dataset_download("columbine/imdb-dataset-sentiment-analysis-in-csv-format")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'imdb-dataset-sentiment-analysis-in-csv-format' dataset.
Path to dataset files: /kaggle/input/imdb-dataset-sentiment-analysis-in-csv-format


In [ ]:
# Find CSV file inside downloaded path
files = os.listdir(path)
print("Files in dataset folder:", files)

# .csv extension
csv_file = [f for f in files if f.endswith('.csv')][1]

df = pd.read_csv(os.path.join(path, csv_file), nrows=10000)

# Display dataset info
print("\nDataset Shape:", df.shape)
print("\nColumns:", df.columns)

# Show first 5 rows
print("\nFirst 5 rows:")
print(df.head())

Files in dataset folder: ['Valid.csv', 'Train.csv', 'Test.csv']

Dataset Shape: (10000, 2)

Columns: Index(['text', 'label'], dtype='object')

First 5 rows:
                                                text  label
0  I grew up (b. 1965) watching and loving the Th...      0
1  When I put this movie in my DVD player, and sa...      0
2  Why do people who do not know what a particula...      0
3  Even though I have great interest in Biblical ...      0
4  Im a die hard Dads Army fan and nothing will e...      1


In [ ]:
# Simple tokenizer
def tokenize(text):
    return text.lower().split()

In [ ]:
PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

# Collect all tokens
all_tokens = [token for text in df['text'] for token in tokenize(text)]

# Count frequency
token_counts = Counter(all_tokens)

# Keep words with freq >= 2
vocab_tokens = [t for t, c in token_counts.items() if c >= 2]

# Build vocab dictionary
vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}

for i, token in enumerate(sorted(vocab_tokens), start=2):
    vocab[token] = i

print(f"Vocabulary size: {len(vocab)}")
print(f"PAD index: {vocab[PAD_TOKEN]}, UNK index: {vocab[UNK_TOKEN]}")

Vocabulary size: 56013
PAD index: 0, UNK index: 1


In [ ]:
MAX_LEN = 300  # fixed sequence length

def text_to_indices(text, vocab, max_len=MAX_LEN):
    tokens = tokenize(text)

    # Map tokens to indices
    indices = [vocab.get(t, vocab[UNK_TOKEN]) for t in tokens]

    # Truncate
    indices = indices[:max_len]

    # Pad
    padding = [vocab[PAD_TOKEN]] * (max_len - len(indices))
    indices = indices + padding

    return indices

# Test conversion
sample = text_to_indices(df['text'].iloc[5555], vocab)
print(f"Sequence length: {len(sample)}")

Sequence length: 300


In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, vocab, max_len=MAX_LEN):
        self.sequences = [text_to_indices(t, vocab, max_len) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        x = torch.tensor(self.sequences[idx], dtype=torch.long)
        y = torch.tensor(self.labels[idx], dtype=torch.float)
        return x, y

In [ ]:
texts = df['text'].tolist()
labels = df['label'].tolist()

X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

train_dataset = SentimentDataset(X_train, y_train, vocab)
test_dataset  = SentimentDataset(X_test, y_test, vocab)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Test batches: {len(test_loader)}")

Train batches: 250
Test batches: 63


In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim,
                 output_dim=1, n_layers=1,
                 bidirectional=False, dropout=0.3, pad_idx=0):
        super().__init__()

        # Embedding layer
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=pad_idx)

        # LSTM layer
        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            num_layers=n_layers,
            batch_first=True,
            bidirectional=bidirectional,
            dropout=dropout if n_layers > 1 else 0
        )

        # Output dimension
        lstm_out_dim = hidden_dim * 2 if bidirectional else hidden_dim

        # Fully connected layer
        self.fc = nn.Linear(lstm_out_dim, output_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x shape: (batch_size, seq_len)

        embedded = self.dropout(self.embedding(x))
        # shape: (batch_size, seq_len, embed_dim)

        output, (hidden, cell) = self.lstm(embedded)

        # Get final hidden state
        if self.lstm.bidirectional:
            hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        else:
            hidden = hidden[-1]

        out = self.fc(self.dropout(hidden)).squeeze(1)
        return out


In [ ]:
VOCAB_SIZE = len(vocab)
EMBED_DIM  = 100
HIDDEN_DIM = 128

model = LSTMClassifier(
    VOCAB_SIZE,
    EMBED_DIM,
    HIDDEN_DIM,
    bidirectional=True
).to(device)

print(model)

LSTMClassifier(
  (embedding): Embedding(56013, 100, padding_idx=0)
  (lstm): LSTM(100, 128, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.3, inplace=False)
)


In [ ]:
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()

    epoch_loss = 0
    correct = 0
    total = 0

    for x_batch, y_batch in loader:
        x_batch = x_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(x_batch)

        loss = criterion(predictions, y_batch)
        loss.backward()

        # Prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()

        epoch_loss += loss.item()

        preds = (torch.sigmoid(predictions) >= 0.5).long()
        correct += (preds == y_batch.long()).sum().item()
        total += y_batch.size(0)

    return epoch_loss / len(loader), correct / total

In [ ]:
N_EPOCHS = 15

for epoch in range(N_EPOCHS):
    loss, acc = train_epoch(model, train_loader, criterion, optimizer, device)
    print(f"Epoch {epoch+1}: Loss = {loss:.4f}, Accuracy = {acc:.4f}")

Epoch 1: Loss = 0.6834, Accuracy = 0.5499
Epoch 2: Loss = 0.6360, Accuracy = 0.6422
Epoch 3: Loss = 0.5665, Accuracy = 0.7130
Epoch 4: Loss = 0.4868, Accuracy = 0.7772
Epoch 5: Loss = 0.3969, Accuracy = 0.8300
Epoch 6: Loss = 0.3305, Accuracy = 0.8594
Epoch 7: Loss = 0.2609, Accuracy = 0.9001
Epoch 8: Loss = 0.2258, Accuracy = 0.9126
Epoch 9: Loss = 0.1788, Accuracy = 0.9321
Epoch 10: Loss = 0.1575, Accuracy = 0.9427
Epoch 11: Loss = 0.1277, Accuracy = 0.9527
Epoch 12: Loss = 0.1059, Accuracy = 0.9606
Epoch 13: Loss = 0.0881, Accuracy = 0.9691
Epoch 14: Loss = 0.0724, Accuracy = 0.9745
Epoch 15: Loss = 0.0611, Accuracy = 0.9800


In [ ]:
def evaluate(model, loader, device):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for x_batch, y_batch in loader:
            x_batch = x_batch.to(device)

            logits = model(x_batch)
            preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()

            all_preds.extend(preds)
            all_labels.extend(y_batch.long().numpy())

    return np.array(all_preds), np.array(all_labels)

In [ ]:
preds, labels = evaluate(model, test_loader, device)

print("\n=== Test Set Evaluation ===")
print(f"Accuracy: {accuracy_score(labels, preds):.4f}")
print(f"F1 Score: {f1_score(labels, preds):.4f}")

print("\nClassification Report:")
print(classification_report(labels, preds, target_names=["negative", "positive"]))


=== Test Set Evaluation ===
Accuracy: 0.7985
F1 Score: 0.7970

Classification Report:
              precision    recall  f1-score   support

    negative       0.83      0.77      0.80      1046
    positive       0.77      0.83      0.80       954

    accuracy                           0.80      2000
   macro avg       0.80      0.80      0.80      2000
weighted avg       0.80      0.80      0.80      2000



**Discussion**

The LSTM model shows strong performance, achieving an accuracy of 79.85% and an F1-score of 0.797. The model is effectively learning sentiment patterns.

There is balanced overall performance

Precision, recall, and F1-scores are all around 0.80.
The model is performing consistently across both classes. Good class-wise performance.


Better at finding positives (higher recall)

Slightly better confidence on negatives (higher precision)

In [ ]:
# !pip install transformers

from transformers import pipeline
import torch

# Device setup
device = 0 if torch.cuda.is_available() else -1
torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f'Using device: {torch_device}')

# Load pipeline
print("Loading sentiment analysis pipeline...")
sentiment_pipe = pipeline('sentiment-analysis', device=device)

print("Pipeline ready.")
print(f"Model used: {sentiment_pipe.model.config._name_or_path}")

No model was supplied, defaulted to distilbert/distilbert-base-uncased-finetuned-sst-2-english and revision 714eb0f.
Using a pipeline without specifying a model name and revision in production is not recommended.


Using device: cuda
Loading sentiment analysis pipeline...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

Pipeline ready.
Model used: distilbert/distilbert-base-uncased-finetuned-sst-2-english


In [ ]:
# Test on real samples
sample_texts = df['text'].sample(5, random_state=42).tolist()

# Run predictions
results = sentiment_pipe(sample_texts)

print("\n=== Dataset Samples ===")
for text, res in zip(sample_texts, results):
    print(f"\nText: {text[:150]}...")
    print(f"Prediction: {res['label']} ({res['score']:.4f})")


=== Dataset Samples ===

Text: I tried. God knows I tried to like this Swiss Cheese of a movie, but the story was too full of holes, some big enough to drive a horse drawn carriage ...
Prediction: NEGATIVE (0.9993)

Text: A sophisticated contemporary fable about the stresses that work to loosen and ultimately unbind the vows of marriage. The main thrust of the narrative...
Prediction: POSITIVE (0.9942)

Text: This brief review contains no spoilers since the movie spoils itself. It is wooden and pedantic. It has no saving grace whatsoever. If someone invites...
Prediction: NEGATIVE (0.9994)

Text: This movie was perhaps the biggest waste of 2 hours of my life. From the opening 10 minutes, I was ready to leave. The cliches there slapping you in t...
Prediction: NEGATIVE (0.9998)

Text: Went to see this movie with my brother and his girlfriend. The place was pretty packed and we all laughed so hard it was easy to miss lines. I knew it...
Prediction: POSITIVE (0.9982)


**1. Ease of Use & Performance vs LSTM**

Requires a single line of code to perform sentiment analysis. LSTM required preprocessing, vocabulary creation, model training, and evaluation.

No training needed (pre-trained model)

Instant predictions

Predictions more accurate and confident


**2. Self-Attention**

Core idea: Each word in a sentence can attend to (focus on) all other words when computing its representation. The model looks at the entire sentence and assigns importance weights to words.

Example: The movie was not good because it was boring.

Self-attention helps the model to connect "not" to "good" to understand negation

Self-attention helps the model to connect "it" to "movie" to understand reference


**3. Difference from RNN/LSTM**

RNN/LSTM processes sequentially, memory passed step by step and slower.

Transformer processes entire sequence at once, direct access to all words and faster(parallelizable)

**4. Advantages**

Handles long-range dependencies: can connect distant words easily

Parallel computation: faster training and inference

Better contextual understanding: considers full sentence context

Pre-trained models like BERT, DistilBERT


**5. RNN vs Transformer (final section)**

**RNN/LSTM**

**Pros**

Simpler architecture

Works well on small datasets

Lower computational cost

**Cons**

Sequential, slow

Struggles with long dependencies

Requires training from scratch

**Transformer**

**Pros**

Better performance

Captures full context

Pre-trained

**Cons**

Computationally expensive

Requires more memory

Less interpretable